In [ ]:

import os
import subprocess
import sys

try:
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaInMemoryUpload
    from google.oauth2.credentials import Credentials
except ImportError:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "google-api-python-client", "google-auth"
    ])
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaInMemoryUpload
    from google.oauth2.credentials import Credentials

# Google Drive destination folder
FOLDER_ID = "YOUR_FOLDER_ID"

# Read credentials from environment variables
CLIENT_ID = os.environ["GOOGLE_CLIENT_ID"]
CLIENT_SECRET = os.environ["GOOGLE_CLIENT_SECRET"]
REFRESH_TOKEN = os.environ["GOOGLE_REFRESH_TOKEN"]

creds = Credentials(
    None,
    refresh_token=REFRESH_TOKEN,
    token_uri="https://oauth2.googleapis.com/token",
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET
)

drive_service = build("drive", "v3", credentials=creds)

code_content = 'print("hello world")\n'

media = MediaInMemoryUpload(
    code_content.encode("utf-8"),
    mimetype="text/x-python"
)

metadata = {
    "name": "hello_world.py",
    "parents": [FOLDER_ID]
}

file = drive_service.files().create(
    body=metadata,
    media_body=media,
    fields="id, name"
).execute()

print(f"SUCCESS: Created '{file['name']}' in Google Drive!")
print(f"File ID: {file['id']}")



[OK] SUCCESS: Created 'hello_world.py' in your Google Drive folder!
File ID: 1cyMEIzY4Ts7NC9wRUNd2P1xDgO0ECOnD
View in Drive: https://drive.google.com/drive/folders/13vd_i1ePxEbYH5RtW9Q4B2OG0ccwv6Wz


**Setup**

In [ ]:
from pathlib import Path
import re, zipfile, os

base = Path("/mnt/data")
cell1_src = base / "Pasted code(20261022-234336).py"
cell2_src = base / "Pasted code (2)(2).py"

# Cell 1 does not contain the OAuth values, so preserve it as supplied.
cell1_out = base / "drive_cell1_trellis_setup.py"
cell1_out.write_text(cell1_src.read_text(encoding="utf-8"), encoding="utf-8")

text = cell2_src.read_text(encoding="utf-8")

# Replace hard-coded OAuth credentials inside BATCH_CODE with environment lookups.
text, n_id = re.subn(
    r'(?m)^CLIENT_ID\s*=\s*["\'][^"\']*["\']\s*$',
    'CLIENT_ID = os.environ["GOOGLE_CLIENT_ID"]',
    text
)
text, n_secret = re.subn(
    r'(?m)^CLIENT_SECRET\s*=\s*["\'][^"\']*["\']\s*$',
    'CLIENT_SECRET = os.environ["GOOGLE_CLIENT_SECRET"]',
    text
)
text, n_refresh = re.subn(
    r'(?m)^REFRESH_TOKEN\s*=\s*["\'][^"\']*["\']\s*$',
    'REFRESH_TOKEN = os.environ["GOOGLE_REFRESH_TOKEN"]',
    text
)

# Add Kaggle Secrets loading in the outer notebook cell before constructing env.
needle = 'env = os.environ.copy()\n'
insert = '''# Load OAuth credentials from Kaggle Secrets (create these three secrets in the Kaggle notebook settings).
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

env = os.environ.copy()
env["GOOGLE_CLIENT_ID"] = user_secrets.get_secret("GOOGLE_CLIENT_ID")
env["GOOGLE_CLIENT_SECRET"] = user_secrets.get_secret("GOOGLE_CLIENT_SECRET")
env["GOOGLE_REFRESH_TOKEN"] = user_secrets.get_secret("GOOGLE_REFRESH_TOKEN")
'''
if needle in text and 'user_secrets.get_secret("GOOGLE_CLIENT_ID")' not in text:
    text = text.replace(needle, insert, 1)

cell2_out = base / "drive_cell2_trellis_batch_SECURE.py"
cell2_out.write_text(text, encoding="utf-8")

# Include both notebook-cell scripts in a zip for convenient transfer.
zip_out = base / "drive_notebook_cells_SECURE.zip"
with zipfile.ZipFile(zip_out, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(cell1_out, arcname=cell1_out.name)
    z.write(cell2_out, arcname=cell2_out.name)

print("Prepared corrected files.")
print(f"OAuth hardcoded assignments replaced: client_id={n_id}, client_secret={n_secret}, refresh_token={n_refresh}")
print("The original credential values were not printed.")
print(f"Files: {cell1_out.name}, {cell2_out.name}, {zip_out.name}")


W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


Reading package lists...
Building dependency tree...
Reading state information...
The following additional packages will be installed:
  cpp-11 gcc-11-base libasan6 libgcc-11-dev libstdc++-11-dev libtsan0
Suggested packages:
  gcc-11-locales g++-11-multilib gcc-11-doc gcc-11-multilib libstdc++-11-doc
The following NEW packages will be installed:
  cpp-11 g++-11 gcc-11 gcc-11-base libasan6 libgcc-11-dev libstdc++-11-dev
  libtsan0
0 upgraded, 8 newly installed, 0 to remove and 207 not upgraded.
Need to get 50.7 MB of archives.
After this operation, 160 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/universe amd64 gcc-11-base amd64 11.5.0-1ubuntu1~24.04.1 [45.6 kB]
Get:2 http://archive.ubuntu.com/ubuntu noble-updates/universe amd64 cpp-11 amd64 11.5.0-1ubuntu1~24.04.1 [9,999 kB]
Get:3 http://archive.ubuntu.com/ubuntu noble-updates/universe amd64 libasan6 amd64 11.5.0-1ubuntu1~24.04.1 [2,264 kB]
Get:4 http://archive.ubuntu.com/ubuntu noble-u

**Project**

In [ ]:
from pathlib import Path
import re

src = Path("/mnt/data/Pasted code(20261022-235025).py")
out = Path("/mnt/data/drive_cell2_trellis_batch_SECURE.py")
text = src.read_text(encoding="utf-8")

patterns = {
    "client_id": (r'(?m)^CLIENT_ID\s*=\s*["\'][^"\']*["\']\s*$', 'CLIENT_ID = os.environ["GOOGLE_CLIENT_ID"]'),
    "client_secret": (r'(?m)^CLIENT_SECRET\s*=\s*["\'][^"\']*["\']\s*$', 'CLIENT_SECRET = os.environ["GOOGLE_CLIENT_SECRET"]'),
    "refresh_token": (r'(?m)^REFRESH_TOKEN\s*=\s*["\'][^"\']*["\']\s*$', 'REFRESH_TOKEN = os.environ["GOOGLE_REFRESH_TOKEN"]'),
}
counts = {}
for name, (pattern, replacement) in patterns.items():
    text, counts[name] = re.subn(pattern, replacement, text)

needle = 'env = os.environ.copy()\n'
insert = '''# Load credentials from Kaggle Secrets and pass them to the TRELLIS subprocess.
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

env = os.environ.copy()
env["GOOGLE_CLIENT_ID"] = user_secrets.get_secret("GOOGLE_CLIENT_ID")
env["GOOGLE_CLIENT_SECRET"] = user_secrets.get_secret("GOOGLE_CLIENT_SECRET")
env["GOOGLE_REFRESH_TOKEN"] = user_secrets.get_secret("GOOGLE_REFRESH_TOKEN")
'''
if 'user_secrets.get_secret("GOOGLE_CLIENT_ID")' not in text:
    if needle not in text:
        raise ValueError("Could not find environment setup block; file was not written.")
    text = text.replace(needle, insert, 1)

out.write_text(text, encoding="utf-8")
print("Created corrected file:", out.name)
print("Replaced hard-coded credential assignments:", counts)
print("Added Kaggle Secrets loading to parent environment.")
